<a href="https://colab.research.google.com/github/juanpajaro/aprendizaje_profundo_salud_puj_2026/blob/main/Taller_1_DL_en_salud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import keras
import pandas as pd
import numpy as np
import re
from keras.preprocessing.sequence import pad_sequences

In [ ]:
datos = pd.read_csv('/content/datos_apnea.csv')
datos.head()

In [ ]:
datos["Apnea"].value_counts()

In [ ]:
#count NaN in datos["Apnea"]
datos["Apnea"].isna().sum()

In [ ]:
#convert NaN in datos["Apnea"] to 1 and "No" to cero
datos["Apnea"] = datos["Apnea"].fillna(1)
datos["Apnea"] = datos["Apnea"].replace("No", 0)
datos["Apnea"].value_counts()

In [ ]:
text_raw = datos["EnfermedadActual"].to_numpy()
print(type(text_raw))
y_raw = datos["Apnea"].to_numpy()
print(type(y_raw))

In [ ]:
print(len(text_raw))
print(text_raw.shape)

In [ ]:
print(text_raw[1])

In [ ]:
def encontrar_palabras(oracion):
  #reemplaza signos de puntuacion por puntos
  oracion_limpia = re.sub(r'[,!?;-]', '.', oracion)
  #convertir la oración en minuscula
  oracion_minuscula = oracion_limpia.lower()
  #eliminar comillas dobles y simples
  oracion_sin_comilla = re.sub(r'["\']', '', oracion_minuscula)
  #se eliminan espacios
  l_oracion = oracion_sin_comilla.split()
  oracion_final = [palabra.strip() for palabra in l_oracion]
  return " ".join(oracion_final)

In [ ]:
#Ejemplo de ejecución y resultado de la función anterior
for i in text_raw[:2]:
  prueba = encontrar_palabras(i)
  print(prueba)

In [ ]:
for i in range(len(text_raw)):
  text_raw[i] = encontrar_palabras(text_raw[i])

print(text_raw[1])

In [ ]:
lp_ora = text_raw[1].split()
print(lp_ora)
print(list(set(lp_ora)))

In [ ]:
def obtener_diccionario(oraciones, verbose=False):
  palabraAindice = {}
  indiceApalabra = {}
  indice = 0
  for oracion in oraciones:
    l_oracion = oracion.split()
    palabras_unicas_en_oracion = list(set(l_oracion))
    if verbose:
      print(l_oracion)
    for palabra in palabras_unicas_en_oracion:
      if palabra not in palabraAindice:
        palabraAindice[palabra] = indice
        indiceApalabra[indice] = palabra
        indice += 1
  return palabraAindice, indiceApalabra

In [ ]:
#Ejemplo de ejecución y resultado de la función anterior
prueba_1, prueba_2 = obtener_diccionario(text_raw[:2], True)
print(prueba_1)
print(prueba_2)
print(len(prueba_1))
print(len(prueba_2))

In [ ]:
#se ejecuta sobre todo el texto
palabraAindice, indiceApalabra = obtener_diccionario(text_raw, False)
print(len(palabraAindice))
print(len(indiceApalabra))

In [ ]:
def convertir_oraciones_indeces(oracion, palabraAindice, verbose=False):
  l_indices = []
  for palabra in oracion.split():
    if verbose:
      print(palabra)
    l_indices = l_indices + [palabraAindice[palabra]]
  return np.array(l_indices)

In [ ]:
#Ejemplo de ejecución y resultado de la función anterior
for i in text_raw[:2]:
  prueba = convertir_oraciones_indeces(i, palabraAindice, False)
  print(prueba)

In [ ]:
sequences = []
for i in range(len(text_raw)):
  texto_por_oracion = convertir_oraciones_indeces(text_raw[i], palabraAindice, verbose=False)
  sequences.append(texto_por_oracion)

# Rellenar secuencias para asegurar una longitud uniforme
# Puedes elegir una longitud máxima (logitud_oracion_max) o determinarla dinámicamente, por ejemplo, max(len(s) para s en secuencias)
# Para este ejemplo, encontremos la longitud máxima entre todas las secuencias
logitud_oracion_max = max(len(s) for s in sequences) if sequences else 0
print(logitud_oracion_max)
todo_texto_indice_padded = pad_sequences(sequences, maxlen=logitud_oracion_max, padding='post')

# Impresion de la primera oracion en indices
print(todo_texto_indice_padded[1])
print(type(todo_texto_indice_padded))

In [ ]:
print(type(todo_texto_indice_padded))
print(todo_texto_indice_padded.shape)
print(type(y_raw))
print(y_raw.shape)

In [ ]:
#Completar y entrenar un modelo de redes neuronales densas de aqui en adelante